# Libraries

In [17]:
import pandas as pd
import ast
import re
from sklearn.preprocessing import MultiLabelBinarizer

# Data Inspection

In this step, all datasets used in the analysis are loaded into the notebook. These include the international job postings dataset, the **Philippine Labor Force Survey (LFS)** dataset, and **O*NET reference files** containing *occupation titles and skills*. The shapes of the datasets are printed to confirm successful loading, and a preview of the job postings dataset is displayed for initial inspection.

In [2]:
# International job postings
jobs = pd.read_csv("data_jobs.csv")

# Local LFS data
lfs = pd.read_csv("FIES2015 - LFSJAN16 CSV - Cleaned.csv")

# Occupation reference files
occupation = pd.read_excel("db_30_2_excel/Occupation Data.xlsx")
titles = pd.read_excel("db_30_2_excel/Alternate Titles.xlsx")
skills = pd.read_excel("db_30_2_excel/Skills.xlsx")
tech_skills = pd.read_excel("db_30_2_excel/Technology Skills.xlsx")
reported_titles = pd.read_excel("db_30_2_excel/Sample of Reported Titles.xlsx")

print(jobs.shape)
print(lfs.shape)

jobs.head()

C:\Users\Raymond\AppData\Local\Temp\ipykernel_16636\2857268186.py:5: DtypeWarning: Columns (101,102,103,104,105,108,109,110,111,112,114,117,118,119,120,122,123,124,125,126,127,128,140,150,151,152) have mixed types. Specify dtype option on import or set low_memory=False.
  lfs = pd.read_csv("FIES2015 - LFSJAN16 CSV - Cleaned.csv")


(785741, 17)
(207212, 154)


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Senior Data Engineer,Senior Clinical Data Engineer / Principal Clin...,"Watertown, CT",via Work Nearby,Full-time,False,"Texas, United States",2023-06-16 13:44:15,False,False,United States,NaN,NaN,NaN,Boehringer Ingelheim,NaN,NaN
1,Data Analyst,Data Analyst,"Guadalajara, Jalisco, Mexico",via BeBee México,Full-time,False,Mexico,2023-01-14 13:18:07,False,False,Mexico,NaN,NaN,NaN,Hewlett Packard Enterprise,"['r', 'python', 'sql', 'nosql', 'power bi', 't...","{'analyst_tools': ['power bi', 'tableau'], 'pr..."
2,Data Engineer,"Data Engineer/Scientist/Analyst, Mid or Senior...","Berlin, Germany",via LinkedIn,Full-time,False,Germany,2023-10-10 13:14:55,False,False,Germany,NaN,NaN,NaN,ALPHA Augmented Services,"['python', 'sql', 'c#', 'azure', 'airflow', 'd...","{'analyst_tools': ['dax'], 'cloud': ['azure'],..."
3,Data Engineer,LEAD ENGINEER - PRINCIPAL ANALYST - PRINCIPAL ...,"San Antonio, TX",via Diversity.com,Full-time,False,"Texas, United States",2023-07-04 13:01:41,True,False,United States,NaN,NaN,NaN,Southwest Research Institute,"['python', 'c++', 'java', 'matlab', 'aws', 'te...","{'cloud': ['aws'], 'libraries': ['tensorflow',..."
4,Data Engineer,Data Engineer- Sr Jobs,"Washington, DC",via Clearance Jobs,Full-time,False,Sudan,2023-08-07 14:29:36,False,False,Sudan,NaN,NaN,NaN,Kristina Daniel,"['bash', 'python', 'oracle', 'aws', 'ansible',...","{'cloud': ['oracle', 'aws'], 'other': ['ansibl..."


In [3]:
jobs.columns

Index(['job_title_short', 'job_title', 'job_location', 'job_via',
       'job_schedule_type', 'job_work_from_home', 'search_location',
       'job_posted_date', 'job_no_degree_mention', 'job_health_insurance',
       'job_country', 'salary_rate', 'salary_year_avg', 'salary_hour_avg',
       'company_name', 'job_skills', 'job_type_skills'],
      dtype='object')

In [4]:
lfs.columns

Index(['id', 'person', 'w_regn', 'other_id', 'rfact', 'pop_adj', 'urb', 'tstr',
       'tpsu', 'pcinc',
       ...
       'high_allobs', 'low_allobs', 'high_allobs_emp', 'low_allobs_emp',
       'psic', 'industrygrp1', 'industrygrp2', 'educgrp1', 'classworkrgrp',
       'agegrp1'],
      dtype='object', length=154)

# Preprocessing the International Dataset

This step creates a working dataset from the international job postings data by selecting only the columns needed for the analysis. The selected columns include job title, country, salary information, and listed skills. A copy of the filtered dataset is created to avoid modifying the original data.

In [5]:
jobs_model = jobs[
[
"job_title_short",
"job_country",
"salary_rate",
"salary_year_avg",
"salary_hour_avg",
"job_skills"
]
].copy()

## Standardizing the Salary

This step standardizes salary values by creating a single yearly salary column. If a job posting already contains an average yearly salary (`salary_year_avg`), **that value is used directly**. If only an hourly salary (`salary_hour_avg`) is available, it is **converted to a yearly salary** by *multiplying it by 40 hours per week and 52 weeks per year*. Rows without salary information are removed to ensure consistent salary values for analysis.

*Note:* The conversion assumes a standard full-time schedule of 40 hours per week and 52 weeks per year.

**Source**
- https://clockify.me/blog/managing-time/how-many-work-hours-in-a-year/

In [6]:
def get_salary(row):

    if pd.notna(row["salary_year_avg"]):
        return row["salary_year_avg"]

    if pd.notna(row["salary_hour_avg"]):
        return row["salary_hour_avg"] * 40 * 52

    return None


jobs_model["salary"] = jobs_model.apply(get_salary, axis=1)

jobs_model = jobs_model.dropna(subset=["salary"])

## Preprocessing the Job Roles

*Note:* No filtering for data industry roles was required because the dataset already consists of job postings related to data and analytics roles. According to the dataset description, it contains real-world data analytics job postings collected from various online job sources. Therefore, all job titles are assumed to belong to the data industry. A count of job titles was performed to examine the distribution of roles in the dataset.

**Source**
- https://huggingface.co/datasets/lukebarousse/data_jobs

In [7]:
jobs_model["job_title_short"].value_counts()

job_title_short
Data Analyst                 9589
Data Scientist               8503
Data Engineer                6777
Senior Data Engineer         2020
Senior Data Scientist        2015
Senior Data Analyst          1484
Business Analyst              998
Machine Learning Engineer     622
Software Engineer             571
Cloud Engineer                 86
Name: count, dtype: int64

This **function** cleans and standardizes the skill data by converting different formats (lists, strings, or missing values) into a consistent list of individual skills. It removes extra characters, splits merged skills, and ensures each job posting returns a clean list of skills.

In [8]:
def clean_skills(x):
    # Case 1: already a proper list → flatten and clean
    if isinstance(x, list):
        all_items = []
        for item in x:
            text = str(item).strip().strip("'\"[]")
            # split on commas in case items are merged
            parts = re.split(r",\s*", text)
            all_items.extend(parts)
        return [s.strip().strip("'\" ").lower() for s in all_items if s.strip().strip("'\" ")]

    # Case 2: None or NaN (scalar only)
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    # Case 3: string → try parsing, then manual cleanup
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                # recurse to handle nested mess
                return clean_skills(parsed)
        except:
            pass
        # fallback: strip brackets and split on commas
        x_clean = x.strip().strip("[]")
        parts = re.split(r",\s*", x_clean)
        return [s.strip().strip("'\" ").lower() for s in parts if s.strip().strip("'\" ")]

    return []

Applying the `clean_skills` function to the `job_skills` column to create a clean and standardized list of skills for each job posting.

In [9]:
jobs_model["skills_list"] = jobs_model["job_skills"].apply(clean_skills)

This step removes duplicate skills within each row while preserving the original order of skills.

In [10]:
jobs_model["skills_list"] = jobs_model["skills_list"].apply(
    lambda x: list(dict.fromkeys(x))
)

This step ensures that all values in `skills_list` are valid lists to avoid errors in later processing.

In [13]:
jobs_model["skills_list"] = jobs_model["skills_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

This dictionary maps countries to broader geographic regions (e.g., NA, EU, SEA). It allows the dataset to group job postings by region for easier comparison and analysis.

In [15]:
region_map = {

    # North America
    "United States": "NA",
    "Canada": "NA",

    # Latin America
    "Mexico": "LATAM",
    "Costa Rica": "LATAM",
    "Honduras": "LATAM",
    "Guatemala": "LATAM",
    "Panama": "LATAM",
    "Bahamas": "LATAM",
    "Jamaica": "LATAM",
    "Puerto Rico": "LATAM",
    "Dominican Republic": "LATAM",
    "Colombia": "LATAM",
    "Brazil": "LATAM",
    "Argentina": "LATAM",
    "Chile": "LATAM",
    "Peru": "LATAM",
    "Uruguay": "LATAM",
    "Bolivia": "LATAM",
    "Paraguay": "LATAM",
    "Venezuela": "LATAM",
    "El Salvador": "LATAM",
    "Nicaragua": "LATAM",

    # Europe
    "United Kingdom": "EU",
    "Germany": "EU",
    "France": "EU",
    "Spain": "EU",
    "Belgium": "EU",
    "Denmark": "EU",
    "Hungary": "EU",
    "Poland": "EU",
    "Portugal": "EU",
    "Sweden": "EU",
    "Estonia": "EU",
    "Croatia": "EU",
    "Netherlands": "EU",
    "Ireland": "EU",
    "Lithuania": "EU",
    "Slovenia": "EU",
    "Austria": "EU",
    "Greece": "EU",
    "Czechia": "EU",
    "Switzerland": "EU",
    "Italy": "EU",
    "Norway": "EU",
    "Luxembourg": "EU",
    "Romania": "EU",
    "Finland": "EU",
    "Latvia": "EU",
    "Slovakia": "EU",
    "Serbia": "EU",
    "Montenegro": "EU",
    "Bosnia and Herzegovina": "EU",
    "Belarus": "EU",
    "Ukraine": "EU",

    # Southeast Asia
    "Philippines": "SEA",
    "Singapore": "SEA",
    "Indonesia": "SEA",
    "Malaysia": "SEA",
    "Thailand": "SEA",
    "Vietnam": "SEA",
    "Cambodia": "SEA",
    "Myanmar": "SEA",
    "Brunei": "SEA",

    # East Asia
    "Japan": "EAST_ASIA",
    "South Korea": "EAST_ASIA",
    "Taiwan": "EAST_ASIA",
    "China": "EAST_ASIA",
    "Hong Kong": "EAST_ASIA",

    # South Asia
    "India": "SOUTH_ASIA",
    "Pakistan": "SOUTH_ASIA",
    "Bangladesh": "SOUTH_ASIA",
    "Sri Lanka": "SOUTH_ASIA",
    "Nepal": "SOUTH_ASIA",

    # Middle East
    "Israel": "ME",
    "United Arab Emirates": "ME",
    "Jordan": "ME",
    "Turkey": "ME",
    "Lebanon": "ME",

    # Africa
    "South Africa": "AFRICA",
    "Nigeria": "AFRICA",
    "Kenya": "AFRICA",
    "Ghana": "AFRICA",
    "Morocco": "AFRICA",
    "Egypt": "AFRICA",
    "Zimbabwe": "AFRICA",
    "Namibia": "AFRICA",
    "Tunisia": "AFRICA",
    "Algeria": "AFRICA",
    "Uganda": "AFRICA",
    "Cameroon": "AFRICA",
    "Senegal": "AFRICA",
    "Mali": "AFRICA",
    "Benin": "AFRICA",
    "Sudan": "AFRICA",
    "Zambia": "AFRICA",

    # Oceania
    "Australia": "OCEANIA",
    "New Zealand": "OCEANIA"
}

This step creates a new `region` column by mapping each country in `job_country` to its corresponding geographic region using the `region_map` dictionary. Countries that are not included in the mapping are assigned the value `"OTHER"` to handle unmatched cases.

In [16]:
jobs_model["region"] = jobs_model["job_country"].map(region_map).fillna("OTHER")

This step converts the list of skills into numerical features using multi-hot encoding. Each skill becomes a column, where 1 indicates the skill is present and 0 indicates it is absent.

In [18]:
mlb = MultiLabelBinarizer()

skills_matrix = mlb.fit_transform(jobs_model["skills_list"])

skills_df = pd.DataFrame(
    skills_matrix,
    columns=mlb.classes_,
    index=jobs_model.index
)

jobs_model = pd.concat([jobs_model, skills_df], axis=1)

**Skill Frequency Inspection**

In this step, the frequency of each skill in the dataset is examined. Because the skills have been multi-hot encoded, summing each column counts how many job postings contain that skill. This allows us to observe the overall distribution of skills across the dataset.

The results show that some skills appear very frequently (such as SQL, Python, Tableau, and R), while many others appear only a few times. A large number of these skills occur in very small counts (for example, appearing only once or a handful of times). These **rare skills** can introduce noise and unnecessarily increase the number of features in the model.

To improve model stability and reduce dimensionality, the next step filters out these rare skills and keeps only those that appear above a minimum frequency threshold.

In [23]:
import pandas as pd
pd.set_option("display.max_rows", None)

skill_counts

sql                18462
python             17680
tableau             7028
r                   6921
aws                 6848
excel               6237
spark               5299
azure               4753
java                3836
power bi            3808
snowflake           3496
hadoop              3125
scala               2512
sas                 2396
nosql               2222
databricks          2211
oracle              2208
kafka               2164
redshift            2001
sql server          1988
go                  1977
airflow             1925
git                 1827
word                1685
gcp                 1593
flow                1587
tensorflow          1584
powerpoint          1505
docker              1462
pandas              1346
pytorch             1320
pyspark             1291
mysql               1240
kubernetes          1149
javascript          1139
looker              1127
shell               1054
c++                 1028
jira                 973
bigquery             968


This step removes rarely occurring skills from the dataset. The frequency of each skill is calculated by summing the multi-hot encoded skill columns. Skills that appear fewer than 50 times are considered rare and are removed, while only commonly occurring skills are retained. This reduces noise and limits the number of features used in the model.

In [24]:
skill_counts = skills_df.sum()

common_skills = skill_counts[skill_counts >= 50].index

skills_df = skills_df[common_skills]

## Training the Model

After filtering the skills to keep only commonly occurring ones, the modeling dataset is rebuilt by combining the filtered skill features with the existing job information in `jobs_model`. This ensures that the dataset used for modeling includes only the selected skill features.

In [27]:
jobs_model = pd.concat([jobs_model, skills_df], axis=1)

This step converts the categorical `region` variable into numerical features using one-hot encoding. Each region becomes a separate binary column indicating whether a job posting belongs to that region. These encoded region variables are then added to the modeling dataset so that geographic location can be used as a feature in the model.

In [28]:
region_dummies = pd.get_dummies(jobs_model["region"], prefix="region")

jobs_model = pd.concat([jobs_model, region_dummies], axis=1)

This step converts the categorical `job_title_short` variable into numerical features using one-hot encoding. Each job title is transformed into a separate binary column indicating whether a job posting belongs to that role. These encoded job title features are then added to the modeling dataset so that the model can capture differences in salary across job roles.

In [39]:
title_dummies = pd.get_dummies(jobs_model["job_title_short"], prefix="title")

jobs_model = pd.concat([jobs_model, title_dummies], axis=1)

This step constructs the feature matrix `X` by removing columns that are not used as model inputs. These include descriptive fields (such as job title and country), intermediate preprocessing columns, and the target variable (`salary`). The remaining columns represent the features used to train the model.

In [40]:
X = jobs_model.drop(columns=[
    "job_title_short",
    "job_country",
    "salary_rate",
    "salary_year_avg",
    "salary_hour_avg",
    "job_skills",
    "skills_list",
    "region",
    "salary"
])

The target variable `y`,

In [41]:
y = jobs_model["salary"]

This step divides the dataset into training and testing sets using `train_test_split`. The feature matrix `X` and target variable `y` are split so that 80% of the data is used to train the model and 20% is reserved for testing. The `random_state=42` parameter ensures that the split is reproducible, meaning the same data partition will be generated each time the code is run.

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [43]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

Initializing the models,

In [44]:
lr = LinearRegression()

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

Training the models,

In [45]:
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,200
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


Making predictions,

In [46]:
lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)
gb_pred = gb.predict(X_test)

Evaluating the models,

In [47]:
from sklearn.metrics import r2_score, mean_absolute_error

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting"],
    "R2": [
        r2_score(y_test, lr_pred),
        r2_score(y_test, rf_pred),
        r2_score(y_test, gb_pred)
    ],
    "MAE": [
        mean_absolute_error(y_test, lr_pred),
        mean_absolute_error(y_test, rf_pred),
        mean_absolute_error(y_test, gb_pred)
    ]
})

results

,Model,R2,MAE
0,Linear Regression,0.231659,31159.130866
1,Random Forest,0.262345,29295.085094
2,Gradient Boosting,0.239454,31019.832968
